In [2]:
import os

DATA_PATH = "/Users/nehaharish/github/Thesis-mlops/data/raw/PaperDataset"
json_files = []

for root, dirs, files in os.walk(DATA_PATH):
    for file in files:
        full_path = os.path.join(root, file)
        if full_path.endswith(".json"):
            json_files.append(full_path)

print(f"Found {len(json_files)} JSON files.")

Found 176 JSON files.


In [3]:
import json, os, re, math
from pathlib import Path
import pandas as pd
import numpy as np
from pprint import pprint
# sanity check one file to see top-level keys or structure
with open(json_files[0], "r") as f:
    sample = json.load(f)

if isinstance(sample, list):
    print("Top-level is a list. Displaying the first element:")
    pprint(sample[0])
elif isinstance(sample, dict):
    print("Top-level keys:", list(sample.keys()))
else:
    print("Top-level is neither a list nor a dictionary. Type:", type(sample))

Top-level is a list. Displaying the first element:
{'camp': {'region_rcv': 'ap-southeast-1',
          'region_snd': 'eu-west-1',
          'size_rcv': 'a4.xlarge',
          'size_snd': 'a4.xlarge'},
 'duration': '300',
 'exp_name': 'nuttcp_19',
 'info_rcv': {'duration': '300',
              'exp_name': 'nuttcp_19',
              'ip_agent': '10.96.214.75',
              'ip_exp_receiver': '104.215.157.206',
              'ip_exp_sender': '191.238.99.208',
              'ip_receiver': '104.215.157.206',
              'ip_sender': '191.238.99.208',
              'port': '18100',
              'protocol': 'tcp',
              'start_time': 1433213619,
              'tool': 'NUTTCP_R'},
 'info_snd': {'duration': '300',
              'exp_name': 'nuttcp_19',
              'granularity': 0.1,
              'ip_agent': '100.79.144.63',
              'ip_exp_receiver': '104.215.157.206',
              'ip_exp_sender': '191.238.99.208',
              'ip_receiver': '104.215.157.206',
        

In [4]:
import json

all_extracted_data = []

for json_file_path in json_files:
    # Infer provider from file path
    provider = "AWS" if "AWS" in json_file_path else ("Azure" if "Azure" in json_file_path else "Unknown")

    with open(json_file_path, "r") as f:
        from_json = json.load(f)

    # Assuming each JSON file contains a list of records
    if isinstance(from_json, list):
        for record in from_json:
            extracted_record = {
                "region_rcv": record.get("camp", {}).get("region_rcv"),
                "region_snd": record.get("camp", {}).get("region_snd"),
                "size_rcv": record.get("camp", {}).get("size_rcv"),
                "size_snd": record.get("camp", {}).get("size_snd"),
                "duration": record.get("duration"),
                "exp_name": record.get("exp_name"),
                "info_rcv": record.get("info_rcv"),
                "info_snd": record.get("info_snd"),
                "proto": record.get("proto"),
                "start": record.get("start"),
                "delay_synt": record.get("results", {}).get("delay", {}).get("synt"),
                "jitter_synt": record.get("results", {}).get("jitter", {}).get("synt"),
                "loss_synt": record.get("results", {}).get("loss", {}).get("synt"),
                "tput_synt": record.get("results", {}).get("tput", {}).get("synt"),
                "route": record.get("results", {}).get("route"),
                 "delay_detailed": record.get("results", {}).get("delay", {}).get("detailed"),
                "loss_detailed": record.get("results", {}).get("loss", {}).get("detailed"),
                "tput_detailed": record.get("results", {}).get("tput", {}).get("detailed"),
                "provider": provider # Add provider information
            }
            all_extracted_data.append(extracted_record)
    elif isinstance(from_json, dict):
         extracted_record = {
                "region_rcv": from_json.get("camp", {}).get("region_rcv"),
                "region_snd": from_json.get("camp", {}).get("region_snd"),
                "size_rcv": from_json.get("camp", {}).get("size_rcv"),
                "size_snd": from_json.get("camp", {}).get("size_snd"),
                "duration": from_json.get("duration"),
                "exp_name": from_json.get("exp_name"),
                "info_rcv": from_json.get("info_rcv"),
                "info_snd": from_json.get("info_snd"),
                "proto": from_json.get("proto"),
                "start": record.get("start"),
                "delay_synt": from_json.get("results", {}).get("delay", {}).get("synt"),
                "jitter_synt": from_json.get("results", {}).get("jitter", {}).get("synt"),
                "loss_synt": from_json.get("results", {}).get("loss", {}).get("synt"),
                "tput_synt": from_json.get("results", {}).get("tput", {}).get("synt"),
                "route": from_json.get("results", {}).get("route"),
                "delay_detailed": from_json.get("results", {}).get("delay", {}).get("detailed"),
                "loss_detailed": from_json.get("results", {}).get("loss", {}).get("detailed"),
                "tput_detailed": from_json.get("results", {}).get("tput", {}).get("detailed"),
                "provider": provider # Add provider information
            }
         all_extracted_data.append(extracted_record)


print(f"Extracted data from {len(all_extracted_data)} records.")

Extracted data from 10336 records.


In [5]:
import pandas as pd

df = pd.DataFrame(all_extracted_data)

# Expand the 'delay_synt' column
delay_synt_df = pd.json_normalize(df['delay_synt'], sep='_')
delay_synt_df.columns = ['delay_' + col for col in delay_synt_df.columns]

# Expand the 'jitter_synt' column
jitter_synt_df = pd.json_normalize(df['jitter_synt'], sep='_')
jitter_synt_df.columns = ['jitter_' + col for col in jitter_synt_df.columns]

# Expand the 'loss_synt' column
loss_synt_df = pd.json_normalize(df['loss_synt'], sep='_')
loss_synt_df.columns = ['loss_' + col for col in loss_synt_df.columns]

# Expand the 'tput_synt' column
tput_synt_df = pd.json_normalize(df['tput_synt'], sep='_')
tput_synt_df.columns = ['tput_' + col for col in tput_synt_df.columns]

# Expand the 'info_rcv' column
info_rcv_df = pd.json_normalize(df['info_rcv'], sep='_')
info_rcv_df.columns = ['info_rcv_' + col for col in info_rcv_df.columns]

# Expand the 'info_snd' column
info_snd_df = pd.json_normalize(df['info_snd'], sep='_')
info_snd_df.columns = ['info_snd_' + col for col in info_snd_df.columns]

# Expand the 'route' column
route_df = pd.json_normalize(df['route'], sep='_')
route_df.columns = ['route_' + col for col in route_df.columns]


# Drop the original nested columns
df = df.drop(columns=['delay_synt', 'jitter_synt', 'loss_synt', 'tput_synt', 'info_rcv', 'info_snd', 'route'])

# Concatenate the new DataFrames with the original DataFrame
df = pd.concat([df, delay_synt_df, jitter_synt_df, loss_synt_df, tput_synt_df, info_rcv_df, info_snd_df, route_df], axis=1)

display(df.head())

,region_rcv,region_snd,size_rcv,size_snd,duration,exp_name,proto,start,delay_detailed,loss_detailed,...,info_snd_ip_agent,info_snd_port,info_snd_granularity,info_snd_duration,info_snd_ip_exp_sender,info_snd_ip_sender,info_snd_tool,info_snd_ip_exp_receiver,route_detailed,route_synt
0,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300,nuttcp_19,tcp,1.433214e+09,None,"{'1433213627': 0.0, '1433213628': 0.0, '143321...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN
1,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300,nuttcp_18,tcp,1.433210e+09,None,"{'1433209928': 0.0, '1433209929': 0.0, '143320...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN
2,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300,nuttcp_17,tcp,1.433206e+09,None,"{'1433206272': 0.0, '1433206273': 0.0, '143320...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN
3,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300,nuttcp_16,tcp,1.433203e+09,None,"{'1433202688': 0.0, '1433202689': 0.0, '143320...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN
4,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300,nuttcp_15,tcp,1.433199e+09,None,"{'1433199104': 0.0, '1433199105': 0.0, '143319...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN


In [6]:
for col in df.columns:
    if 'detailed' not in col:
        print(f"\nUnique values in {col}:")
        try:
            # Attempt to print unique values, handle potential errors with unhashable types
            print(df[col].unique())
        except TypeError:
            print(f"  Could not print unique values for {col} due to unhashable types.")


Unique values in region_rcv:
['ap-southeast-1' 'eu-west-1' 'us-east-1' 'sa-east-1' None 'eu-west-1a'
 'ap-southeast-1a' 'ap-southeast-1b' 'eu-west-1c' 'eu-west-1b'
 'us-east-1d' 'us-east-1a' 'us-east-1b' 'sa-east-1a' 'sa-east-1c'
 'sa-east-1b']

Unique values in region_snd:
['eu-west-1' 'ap-southeast-1' 'us-east-1' 'sa-east-1' None
 'ap-southeast-1a' 'eu-west-1c' 'eu-west-1b' 'eu-west-1a'
 'ap-southeast-1b' 'us-east-1d' 'us-east-1a' 'us-east-1b' 'sa-east-1a'
 'sa-east-1c' 'sa-east-1b']

Unique values in size_rcv:
['a4.xlarge' 'a2.medium' None 'm3.medium' 'm3.xlarge' 'medium']

Unique values in size_snd:
['a4.xlarge' 'a2.medium' None 'm3.medium' 'm3.xlarge' 'medium']

Unique values in duration:
['300' None '120' '480']

Unique values in exp_name:
['nuttcp_19' 'nuttcp_18' 'nuttcp_17' 'nuttcp_16' 'nuttcp_15' 'nuttcp_14'
 'nuttcp_13' 'nuttcp_12' 'nuttcp_11' 'nuttcp_10' 'nuttcp_22' 'nuttcp_23'
 'nuttcp_20' 'nuttcp_21' 'nuttcp_24' 'nuttcp_9' 'nuttcp_8' 'nuttcp_3'
 'nuttcp_2' 'nuttcp_1' 'nut

In [7]:
non_detailed_cols = [col for col in df.columns if 'detailed' not in col]

In [8]:
column_summary = []

for col in non_detailed_cols:
    num_unique = df[col].nunique()
    sample_unique = df[col].unique()[:5].tolist()  # Get the first 5 unique values as a list
    column_summary.append({
        'Column Name': col,
        'Number of Unique Values': num_unique,
        'Sample Unique Values': sample_unique
    })

In [9]:
summary_df = pd.DataFrame(column_summary)
display(summary_df)

,Column Name,Number of Unique Values,Sample Unique Values
0,region_rcv,15,"[ap-southeast-1, eu-west-1, us-east-1, sa-east..."
1,region_snd,15,"[eu-west-1, ap-southeast-1, us-east-1, sa-east..."
2,size_rcv,5,"[a4.xlarge, a2.medium, None, m3.medium, m3.xla..."
3,size_snd,5,"[a4.xlarge, a2.medium, None, m3.medium, m3.xla..."
4,duration,3,"[300, None, 120, 480]"
...,...,...,...
58,info_snd_ip_exp_sender,151,"[191.238.99.208, 104.215.157.206, 191.237.20.8..."
59,info_snd_ip_sender,151,"[191.238.99.208, 104.215.157.206, 191.237.20.8..."
60,info_snd_tool,2,"[NUTTCP_S, NUTTCP_R, nan]"
61,info_snd_ip_exp_receiver,151,"[104.215.157.206, 191.238.99.208, 104.41.209.1..."


In [10]:
import pandas as pd

df = pd.DataFrame(all_extracted_data)

In [11]:
df.shape

(10336, 19)

In [12]:
df[df.duration.isna()]

,region_rcv,region_snd,size_rcv,size_snd,duration,exp_name,info_rcv,info_snd,proto,start,delay_synt,jitter_synt,loss_synt,tput_synt,route,delay_detailed,loss_detailed,tput_detailed,provider
856,None,None,None,None,None,paris_1,None,None,None,NaN,None,None,None,None,None,None,None,None,AWS
858,None,None,None,None,None,paris_2,None,None,None,NaN,None,None,None,None,None,None,None,None,AWS
859,None,None,None,None,None,paris_3,None,None,None,NaN,None,None,None,None,None,None,None,None,AWS
860,None,None,None,None,None,paris_4,None,None,None,NaN,None,None,None,None,None,None,None,None,AWS
862,None,None,None,None,None,paris_5,None,None,None,NaN,None,None,None,None,None,None,None,None,AWS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1691,None,None,None,None,None,paris_26,None,None,None,NaN,None,None,None,None,None,None,None,None,AWS
1692,None,None,None,None,None,paris_27,None,None,None,NaN,None,None,None,None,None,None,None,None,AWS
1693,None,None,None,None,None,paris_28,None,None,None,NaN,None,None,None,None,None,None,None,None,AWS
1694,None,None,None,None,None,paris_29,None,None,None,NaN,None,None,None,None,None,None,None,None,AWS


In [13]:
df.dropna(subset=['duration'], inplace=True)
print(f"Shape of DataFrame after dropping rows with NaN in 'duration': {df.shape}")

Shape of DataFrame after dropping rows with NaN in 'duration': (9616, 19)


**Checking for percentage of rows for each unique duration **

In [14]:
duration_percentages = df['duration'].value_counts(normalize=True) * 100
print("Percentage of rows for each unique duration:")
print(duration_percentages.to_string())

Percentage of rows for each unique duration:
duration
300    98.752080
120     0.748752
480     0.499168


In [15]:
df = df[~df['duration'].isin(['120', '480'])].copy()
print(f"Shape of DataFrame after dropping rows with duration '120' and '480': {df.shape}")

Shape of DataFrame after dropping rows with duration '120' and '480': (9496, 19)


In [16]:
print(df['duration'].unique())

['300']


In [17]:
for column in df.columns:
    if column.startswith('delay_'):
        print(f"Unique values in column '{column}':")
        # Check if the column contains dictionaries
        if any(isinstance(x, dict) for x in df[column].dropna()):
            print(df[column].apply(lambda x: str(x) if isinstance(x, dict) else x).unique())
        else:
            print(df[column].unique())
        print("\n" + "-"*50 + "\n")

Unique values in column 'delay_synt':
["{'std': None, 'pctl5': None, 'min': None, 'max': None, 'pctl25': None, 'pctl95': None, 'median': None, 'pctl75': None, 'mean': 316.1}"
 "{'std': None, 'pctl5': None, 'min': None, 'max': None, 'pctl25': None, 'pctl95': None, 'median': None, 'pctl75': None, 'mean': 315.48}"
 "{'std': None, 'pctl5': None, 'min': None, 'max': None, 'pctl25': None, 'pctl95': None, 'median': None, 'pctl75': None, 'mean': 320.69}"
 ...
 "{'std': None, 'pctl5': None, 'min': None, 'max': None, 'pctl25': None, 'pctl95': None, 'median': None, 'pctl75': None, 'mean': 222.95}"
 "{'std': None, 'pctl5': None, 'min': None, 'max': None, 'pctl25': None, 'pctl95': None, 'median': None, 'pctl75': None, 'mean': 230.64}"
 "{'std': None, 'pctl5': None, 'min': None, 'max': None, 'pctl25': None, 'pctl95': None, 'median': None, 'pctl75': None, 'mean': 230.99}"]

--------------------------------------------------

Unique values in column 'delay_detailed':
[None]

--------------------------

In [18]:
columns_to_drop = [col for col in df.columns if col.startswith('delay_') and col != 'delay_mean']
df.drop(columns=columns_to_drop, inplace=True)
print(f"Shape of DataFrame after dropping 'delay_' columns (excluding 'delay_mean'): {df.shape}")

Shape of DataFrame after dropping 'delay_' columns (excluding 'delay_mean'): (9496, 17)


In [19]:
print(df.shape)

(9496, 17)


In [20]:
import pandas as pd

# Recreate df from all_extracted_data to ensure a fresh start
df = pd.DataFrame(all_extracted_data)

# Expand the 'delay_synt' column
delay_synt_df = pd.json_normalize(df['delay_synt'], sep='_')
delay_synt_df.columns = ['delay_' + col for col in delay_synt_df.columns]

# Expand the 'jitter_synt' column
jitter_synt_df = pd.json_normalize(df['jitter_synt'], sep='_')
jitter_synt_df.columns = ['jitter_' + col for col in jitter_synt_df.columns]

# Expand the 'loss_synt' column
loss_synt_df = pd.json_normalize(df['loss_synt'], sep='_')
loss_synt_df.columns = ['loss_' + col for col in loss_synt_df.columns]

# Expand the 'tput_synt' column
tput_synt_df = pd.json_normalize(df['tput_synt'], sep='_')
tput_synt_df.columns = ['tput_' + col for col in tput_synt_df.columns]

# Expand the 'info_rcv' column
info_rcv_df = pd.json_normalize(df['info_rcv'], sep='_')
info_rcv_df.columns = ['info_rcv_' + col for col in info_rcv_df.columns]

# Expand the 'info_snd' column
info_snd_df = pd.json_normalize(df['info_snd'], sep='_')
info_snd_df.columns = ['info_snd_' + col for col in info_snd_df.columns]

# Expand the 'route' column
route_df = pd.json_normalize(df['route'], sep='_')
route_df.columns = ['route_' + col for col in route_df.columns]

# Drop the original nested columns (only synt are expanded, detailed remain in their original column type)
df = df.drop(columns=['delay_synt', 'jitter_synt', 'loss_synt', 'tput_synt', 'info_rcv', 'info_snd', 'route'])

# Concatenate the new DataFrames with the original DataFrame
df = pd.concat([df, delay_synt_df, jitter_synt_df, loss_synt_df, tput_synt_df, info_rcv_df, info_snd_df, route_df], axis=1)

print(f"Shape after expanding nested JSON columns: {df.shape}")

Shape after expanding nested JSON columns: (10336, 67)


In [21]:
# Convert duration to numeric and drop rows with NaN values in 'duration'
df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
df.dropna(subset=['duration'], inplace=True)
print(f"Shape after dropping NaN in 'duration': {df.shape}")

# Drop rows where duration is '120' or '480'
df = df[~df['duration'].isin([120, 480])].copy()
print(f"Shape after dropping duration '120' and '480': {df.shape}")

Shape after dropping NaN in 'duration': (9616, 67)
Shape after dropping duration '120' and '480': (9496, 67)


In [22]:
# Drop all 'delay_' columns except 'delay_mean'
columns_to_drop = [col for col in df.columns if col.startswith('delay_') and col != 'delay_mean']
df.drop(columns=columns_to_drop, inplace=True)
print(f"Final shape of DataFrame: {df.shape}")

Final shape of DataFrame: (9496, 58)


In [23]:
display(df.head())

,region_rcv,region_snd,size_rcv,size_snd,duration,exp_name,proto,start,loss_detailed,tput_detailed,...,info_snd_ip_agent,info_snd_port,info_snd_granularity,info_snd_duration,info_snd_ip_exp_sender,info_snd_ip_sender,info_snd_tool,info_snd_ip_exp_receiver,route_detailed,route_synt
0,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300.0,nuttcp_19,tcp,1.433214e+09,"{'1433213627': 0.0, '1433213628': 0.0, '143321...","{'1433213627': 1.429890909090909, '1433213628'...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN
1,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300.0,nuttcp_18,tcp,1.433210e+09,"{'1433209928': 0.0, '1433209929': 0.0, '143320...","{'1433209928': 1.4296272727272727, '1433209929...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN
2,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300.0,nuttcp_17,tcp,1.433206e+09,"{'1433206272': 0.0, '1433206273': 0.0, '143320...","{'1433206272': 62.921569999999996, '1433206273...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN
3,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300.0,nuttcp_16,tcp,1.433203e+09,"{'1433202688': 0.0, '1433202689': 0.0, '143320...","{'1433202688': 63.43679999999999, '1433202689'...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN
4,ap-southeast-1,eu-west-1,a4.xlarge,a4.xlarge,300.0,nuttcp_15,tcp,1.433199e+09,"{'1433199104': 0.0, '1433199105': 0.0, '143319...","{'1433199104': 65.52969, '1433199105': 71.3083...",...,100.79.144.63,18100,0.1,300,191.238.99.208,191.238.99.208,NUTTCP_S,104.215.157.206,NaN,NaN


In [24]:
for col_name in ['route_detailed', 'route_synt']:
    if col_name in df.columns:
        print(f"Unique values in column '{col_name}':")
        # Check if the column contains dictionaries before trying to make them unique
        if any(isinstance(x, dict) for x in df[col_name].dropna()):
            print(df[col_name].apply(lambda x: str(x) if isinstance(x, dict) else x).unique())
        else:
            print(df[col_name].unique())
        print("\n" + "-"*50 + "\n")
    else:
        print(f"Column '{col_name}' not found in DataFrame.\n" + "-"*50 + "\n")

Unique values in column 'route_detailed':
[nan]

--------------------------------------------------

Unique values in column 'route_synt':
[nan]

--------------------------------------------------



In [25]:
columns_to_drop_route = ['route_detailed', 'route_synt']
df.drop(columns=columns_to_drop_route, inplace=True)
print(f"Shape of DataFrame after dropping 'route_detailed' and 'route_synt': {df.shape}")

Shape of DataFrame after dropping 'route_detailed' and 'route_synt': (9496, 56)


In [26]:
print(f"Unique values in column 'region_rcv':\n{df['region_rcv'].unique()}\n")
print(f"Unique values in column 'region_snd':\n{df['region_snd'].unique()}")

Unique values in column 'region_rcv':
['ap-southeast-1' 'eu-west-1' 'us-east-1' 'sa-east-1' 'eu-west-1a'
 'ap-southeast-1a' 'ap-southeast-1b' 'eu-west-1c' 'eu-west-1b'
 'us-east-1d' 'us-east-1a' 'us-east-1b' 'sa-east-1a' 'sa-east-1c'
 'sa-east-1b']

Unique values in column 'region_snd':
['eu-west-1' 'ap-southeast-1' 'us-east-1' 'sa-east-1' 'ap-southeast-1a'
 'eu-west-1c' 'eu-west-1b' 'eu-west-1a' 'ap-southeast-1b' 'us-east-1d'
 'us-east-1a' 'us-east-1b' 'sa-east-1a' 'sa-east-1c' 'sa-east-1b']


In [27]:
print(df.columns.tolist())

['region_rcv', 'region_snd', 'size_rcv', 'size_snd', 'duration', 'exp_name', 'proto', 'start', 'loss_detailed', 'tput_detailed', 'provider', 'delay_mean', 'loss_std', 'loss_pctl5', 'loss_min', 'loss_max', 'loss_pctl25', 'loss_pctl95', 'loss_median', 'loss_pctl75', 'loss_mean', 'tput_std', 'tput_pctl5', 'tput_min', 'tput_max', 'tput_pctl25', 'tput_pctl95', 'tput_median', 'tput_pctl75', 'tput_mean', 'info_rcv_protocol', 'info_rcv_ip_receiver', 'info_rcv_exp_name', 'info_rcv_start_time', 'info_rcv_ip_agent', 'info_rcv_port', 'info_rcv_duration', 'info_rcv_ip_exp_sender', 'info_rcv_ip_sender', 'info_rcv_tool', 'info_rcv_ip_exp_receiver', 'info_rcv_target_bwd', 'info_rcv_granularity', 'info_snd_protocol', 'info_snd_ip_receiver', 'info_snd_exp_name', 'info_snd_start_time', 'info_snd_target_bwd', 'info_snd_ip_agent', 'info_snd_port', 'info_snd_granularity', 'info_snd_duration', 'info_snd_ip_exp_sender', 'info_snd_ip_sender', 'info_snd_tool', 'info_snd_ip_exp_receiver']


In [28]:
import re
import pandas as pd
import numpy as np

AWS_REGION_RE = re.compile(r"^(?P<pfx>[a-z]{2})-(?P<area>[a-z]+)-(?P<num>\d+)(?P<az>[a-z])?$")

def split_region_aws(x: str):
    """Return (base_region, az) or (x, None) if pattern doesn't match."""
    if not isinstance(x, str) or not x.strip():
        return (np.nan, None)
    x = x.strip().lower()
    m = AWS_REGION_RE.match(x)
    if not m:
        # leave non-AWS-style labels untouched as base, no az
        return (x, None)
    base = f"{m.group('pfx')}-{m.group('area')}-{m.group('num')}"
    az = m.group('az')
    return (base, az)

def continent_from_region(base: str):
    if not isinstance(base, str): return np.nan
    if base.startswith("us-"): return "americas"
    if base.startswith("sa-"): return "americas"
    if base.startswith("eu-"): return "europe"
    if base.startswith("ap-"): return "asia-pacific"
    if base.startswith("af-"): return "africa"
    if base.startswith("me-"): return "middle-east"
    return "other"

# Apply to your columns
for col in ["region_snd", "region_rcv"]:
    base_col = col + "_base"
    az_col   = ("az_snd" if col=="region_snd" else "az_rcv")
    bases, azs = [], []
    for v in df[col].astype("string"):
        b, a = split_region_aws(v if v is not pd.NA else None)
        bases.append(b); azs.append(a)
    df[base_col] = pd.Series(bases, dtype="string")
    df[az_col]   = pd.Series(azs, dtype="string")

# Helpful booleans and pair key (use *_base)
df["same_region"] = df["region_snd_base"].eq(df["region_rcv_base"])

df["snd_continent"] = df["region_snd_base"].apply(continent_from_region).astype("string")
df["rcv_continent"] = df["region_rcv_base"].apply(continent_from_region).astype("string")
df["same_continent"] = df["snd_continent"].eq(df["rcv_continent"])

# Stable pair key for grouping/splits
if "provider" in df.columns:
    df["pair_base"] = (
        df["provider"].astype("string") + ":" +
        df["region_snd_base"].astype("string").fillna('UNKNOWN') + "->" +
        df["region_rcv_base"].astype("string").fillna('UNKNOWN')
    )
else:
    df["pair_base"] = (
        df["region_snd_base"].astype("string").fillna('UNKNOWN') + "->" +
        df["region_rcv_base"].astype("string").fillna('UNKNOWN')
    )

In [29]:
print("Unique (raw) region_snd:", df["region_snd"].nunique())
print("Unique (base) region_snd_base:", df["region_snd_base"].nunique())

print(df[["region_snd","region_snd_base","az_snd"]].drop_duplicates().head(10))
print(df[["region_rcv","region_rcv_base","az_rcv"]].drop_duplicates().head(10))


Unique (raw) region_snd: 15
Unique (base) region_snd_base: 4
           region_snd region_snd_base az_snd
0           eu-west-1       eu-west-1   <NA>
24     ap-southeast-1  ap-southeast-1   <NA>
48          us-east-1       us-east-1   <NA>
112         sa-east-1       sa-east-1   <NA>
1696  ap-southeast-1a       eu-west-1      a
1720  ap-southeast-1a       eu-west-1      b
1792       eu-west-1c       eu-west-1      b
1816       eu-west-1c       us-east-1      a
1888       eu-west-1b       us-east-1      a
1912       eu-west-1b       us-east-1      b
           region_rcv region_rcv_base az_rcv
0      ap-southeast-1  ap-southeast-1   <NA>
24          eu-west-1       eu-west-1   <NA>
68          us-east-1       us-east-1   <NA>
88          sa-east-1       sa-east-1   <NA>
1696       eu-west-1a       us-east-1      a
1720       eu-west-1a       us-east-1      b
1792  ap-southeast-1a       us-east-1      b
1816  ap-southeast-1a       eu-west-1      a
1888  ap-southeast-1b       eu-west-1  

In [30]:
print(df.columns.tolist())


['region_rcv', 'region_snd', 'size_rcv', 'size_snd', 'duration', 'exp_name', 'proto', 'start', 'loss_detailed', 'tput_detailed', 'provider', 'delay_mean', 'loss_std', 'loss_pctl5', 'loss_min', 'loss_max', 'loss_pctl25', 'loss_pctl95', 'loss_median', 'loss_pctl75', 'loss_mean', 'tput_std', 'tput_pctl5', 'tput_min', 'tput_max', 'tput_pctl25', 'tput_pctl95', 'tput_median', 'tput_pctl75', 'tput_mean', 'info_rcv_protocol', 'info_rcv_ip_receiver', 'info_rcv_exp_name', 'info_rcv_start_time', 'info_rcv_ip_agent', 'info_rcv_port', 'info_rcv_duration', 'info_rcv_ip_exp_sender', 'info_rcv_ip_sender', 'info_rcv_tool', 'info_rcv_ip_exp_receiver', 'info_rcv_target_bwd', 'info_rcv_granularity', 'info_snd_protocol', 'info_snd_ip_receiver', 'info_snd_exp_name', 'info_snd_start_time', 'info_snd_target_bwd', 'info_snd_ip_agent', 'info_snd_port', 'info_snd_granularity', 'info_snd_duration', 'info_snd_ip_exp_sender', 'info_snd_ip_sender', 'info_snd_tool', 'info_snd_ip_exp_receiver', 'region_snd_base', 'az_

In [31]:
# 1) use base regions for grouping everywhere
PAIR_KEYS = ["provider","region_snd_base","region_rcv_base"]  # was: provider + raw regions

# 2) when selecting features for models, prefer base + optional AZs
BASE_CATS = ["provider","region_snd_base","region_rcv_base","proto","size_snd","size_rcv"]
OPTIONAL_CATS = ["az_snd","az_rcv","same_region","same_continent"]  # include if you want

# Example: before fitting a model, drop raw region cols to avoid redundancy
def select_model_X(df, extra_numeric=None, include_az=True):
    drop_dupes = ["region_snd","region_rcv"]  # raw columns
    X = df.drop(columns=[c for c in drop_dupes if c in df.columns], errors="ignore").copy()
    # choose categoricals
    cats = BASE_CATS + (OPTIONAL_CATS if include_az else [])
    cats = [c for c in cats if c in X.columns]
    # make sure dtype is string/bool for OHE
    for c in cats:
        if c in X.columns:
            if X[c].dtype.name not in ("string","bool"):
                X[c] = X[c].astype("string")
    # numeric extras if you pass them (e.g., lags)
    if extra_numeric:
        for c in extra_numeric:
            if c in X.columns:
                X[c] = pd.to_numeric(X[c], errors="coerce")
    return X


In [32]:
# 1) confirm collapse worked
print("pairs (raw):", df.groupby(["provider","region_snd","region_rcv"]).size().shape[0])
print("pairs (base):", df.groupby(["provider","region_snd_base","region_rcv_base"]).size().shape[0])

# 2) ensure no AZs leaked into grouping keys
assert df["region_snd_base"].str.contains(r"[a-z]$", na=False).sum() == 0
assert df["region_rcv_base"].str.contains(r"[a-z]$", na=False).sum() == 0

# 3) check data per pair (should increase after collapsing)
by_pair = df.groupby(["provider","region_snd_base","region_rcv_base"]).size().rename("n").sort_values(ascending=False)
by_pair.head(10)


pairs (raw): 52
pairs (base): 24


provider  region_snd_base  region_rcv_base
AWS       us-east-1        sa-east-1          864
          sa-east-1        us-east-1          864
          us-east-1        eu-west-1          768
          eu-west-1        us-east-1          696
          ap-southeast-1   eu-west-1          576
          sa-east-1        eu-west-1          576
          ap-southeast-1   sa-east-1          576
          us-east-1        ap-southeast-1     576
          sa-east-1        ap-southeast-1     576
          eu-west-1        sa-east-1          576
Name: n, dtype: int64

In [33]:
print(df['provider'].unique())

['Azure' 'AWS']


In [34]:
provider_percentages = df['provider'].value_counts(normalize=True) * 100
print("Percentage of rows for each unique provider:")
print(provider_percentages.to_string())

Percentage of rows for each unique provider:
provider
AWS      90.985678
Azure     9.014322


In [35]:
print(f"Shape of DataFrame : {df.shape}")

Shape of DataFrame : (9496, 65)


In [36]:
columns_to_drop_patterns = [
    "info_rcv_ip_", "info_snd_ip_", "info_rcv_port", "info_snd_port",
    "info_rcv_exp_name", "info_snd_exp_name",
    "info_rcv_start_time", "info_snd_start_time",
    "info_rcv_tool", "info_snd_tool",
    "info_rcv_protocol", "info_snd_protocol",
    "info_rcv_granularity", "info_snd_granularity",
    "info_rcv_target_bwd", "info_snd_target_bwd", "route_"
]

columns_to_drop = []
for col in df.columns:
    for pattern in columns_to_drop_patterns:
        if pattern.endswith('_') and col.startswith(pattern): # Handle prefixes
            columns_to_drop.append(col)
            break # Found a match for this column, move to next
        elif col == pattern: # Handle exact matches
            columns_to_drop.append(col)
            break # Found a match for this column, move to next

# Ensure only unique columns are in the list to drop
columns_to_drop = list(set(columns_to_drop))

# Drop the identified columns
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

print(f"Shape of DataFrame after dropping specified columns: {df.shape}")


Shape of DataFrame after dropping specified columns: (9496, 41)


In [37]:
print(df.columns.tolist())

['region_rcv', 'region_snd', 'size_rcv', 'size_snd', 'duration', 'exp_name', 'proto', 'start', 'loss_detailed', 'tput_detailed', 'provider', 'delay_mean', 'loss_std', 'loss_pctl5', 'loss_min', 'loss_max', 'loss_pctl25', 'loss_pctl95', 'loss_median', 'loss_pctl75', 'loss_mean', 'tput_std', 'tput_pctl5', 'tput_min', 'tput_max', 'tput_pctl25', 'tput_pctl95', 'tput_median', 'tput_pctl75', 'tput_mean', 'info_rcv_duration', 'info_snd_duration', 'region_snd_base', 'az_snd', 'region_rcv_base', 'az_rcv', 'same_region', 'snd_continent', 'rcv_continent', 'same_continent', 'pair_base']


In [38]:
import pandas as pd

# Convert 'start' column from Unix timestamp to datetime format
df['start'] = pd.to_datetime(df['start'], unit='s')
print("Converted 'start' column to datetime format.")

# Define the columns for sorting
sort_cols = ['pair_base', 'start']

# Sort the DataFrame and reset the index
df = df.sort_values(by=sort_cols).reset_index(drop=True)

print("DataFrame sorted by pair_base and start column.")
display(df.head())

Converted 'start' column to datetime format.
DataFrame sorted by pair_base and start column.


,region_rcv,region_snd,size_rcv,size_snd,duration,exp_name,proto,start,loss_detailed,tput_detailed,...,info_snd_duration,region_snd_base,az_snd,region_rcv_base,az_rcv,same_region,snd_continent,rcv_continent,same_continent,pair_base
0,ap-southeast-1b,us-east-1b,m3.medium,m3.medium,300.0,nuttcp_1,tcp,2015-04-07 19:23:31,"{'1428434611': 0.0, '1428434612': 0.0, '142843...","{'1428434611': 2.860081818181818, '1428434612'...",...,300,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AWS:UNKNOWN->UNKNOWN
1,ap-southeast-1a,us-east-1a,m3.medium,m3.medium,300.0,nuttcp_1,tcp,2015-04-07 19:25:02,"{'1428434944': 0.0, '1428434945': 0.0, '142843...","{'1428434944': 55.82216999999999, '1428434945'...",...,300,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AWS:UNKNOWN->UNKNOWN
2,sa-east-1a,eu-west-1a,m3.medium,m3.medium,300.0,nuttcp_1,tcp,2015-04-07 19:25:14,"{'1428434944': 0.0, '1428434945': 0.0, '142843...","{'1428434944': 63.96273, '1428434945': 64.3537...",...,300,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AWS:UNKNOWN->UNKNOWN
3,sa-east-1a,eu-west-1b,m3.medium,m3.medium,300.0,nuttcp_1,tcp,2015-04-07 19:25:39,"{'1428434944': 0.0, '1428434945': 0.0, '142843...","{'1428434944': 63.438750000000006, '1428434945...",...,300,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AWS:UNKNOWN->UNKNOWN
4,ap-southeast-1a,us-east-1d,m3.medium,m3.medium,300.0,nuttcp_1,tcp,2015-04-07 19:25:58,"{'1428434944': 0.0, '1428434945': 0.0, '142843...","{'1428434944': 54.525909999999996, '1428434945...",...,300,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AWS:UNKNOWN->UNKNOWN


In [39]:
print(df.shape, df.memory_usage(deep=True).sum()/1e6, 'MB')

(9496, 41) 135.90362 MB


In [40]:
print(df.dtypes.value_counts())


float64           20
object            11
string[python]     7
boolean            2
datetime64[ns]     1
Name: count, dtype: int64


In [41]:
# A1) Keep only summary features in base
keep = [
    "provider","region_snd_base","region_rcv_base","proto","size_snd","size_rcv",
    "tput_mean","tput_std","tput_min","tput_max","tput_pctl5","tput_pctl25","tput_median","tput_pctl75","tput_pctl95",
    "delay_mean","loss_mean","loss_std","loss_min","loss_max","loss_pctl5","loss_pctl25","loss_median","loss_pctl75","loss_pctl95",
    "start","az_snd","az_rcv","snd_continent","rcv_continent","duration"
]
keep = [c for c in keep if c in df.columns]
base_summ = df[keep].copy()

# A2) Save the summaries fast
for c in base_summ.select_dtypes(include="object").columns:
    base_summ[c] = base_summ[c].astype("category")
base_summ.to_parquet("prepared/base_clean.parquet", engine="pyarrow", compression="snappy", index=False)


In [47]:
from pathlib import Path
import pandas as pd

# identify series-like columns (columns that contain dicts mapping t -> value)
series_cols = [c for c in df.columns if df[c].dropna().apply(lambda x: isinstance(x, dict)).any()]

if not series_cols:
    raise RuntimeError("No nested series columns found to explode.")

# choose partition columns (fallback to raw region names if base columns missing)
provider_col = "provider" if "provider" in df.columns else None
snd_col = "region_snd_base" if "region_snd_base" in df.columns else ("region_snd" if "region_snd" in df.columns else None)
rcv_col = "region_rcv_base" if "region_rcv_base" in df.columns else ("region_rcv" if "region_rcv" in df.columns else None)

partition_cols = [c for c in (provider_col, snd_col, rcv_col) if c is not None]
if len(partition_cols) < 1:
    raise RuntimeError("At least one partition column required (provider/region_snd_base/region_rcv_base).")

records = []
# use reset_index to get a stable run_id
for run_id, row in df.reset_index().iterrows():
    meta = {
        "run_id": int(run_id),
        provider_col: row[provider_col] if provider_col else None,
        snd_col: row[snd_col] if snd_col else None,
        rcv_col: row[rcv_col] if rcv_col else None,
    }
    for col in series_cols:
        series_name = col.replace("_detailed", "").replace("_synt", "").rstrip("_")
        d = row[col]
        if not isinstance(d, dict):
            continue
        for t_key, val in d.items():
            # try to coerce timestamp-like keys to int then to datetime; keep original if not possible
            try:
                t_ts = pd.to_datetime(int(t_key), unit="s")
            except Exception:
                try:
                    t_ts = pd.to_datetime(t_key)
                except Exception:
                    t_ts = t_key
            records.append({
                "run_id": meta["run_id"],
                "series": series_name,
                "t": t_ts,
                "value": val,
                **({provider_col: meta[provider_col]} if provider_col else {}),
                **({snd_col: meta[snd_col]} if snd_col else {}),
                **({rcv_col: meta[rcv_col]} if rcv_col else {}),
            })

# build long DataFrame
series_long = pd.DataFrame.from_records(records)

# ensure columns order
cols_order = ["run_id", "series", "t", "value"] + [c for c in partition_cols if c in series_long.columns]
series_long = series_long[cols_order]

# create prepared folder and save partitioned parquet
Path("prepared").mkdir(parents=True, exist_ok=True)
series_long.to_parquet("prepared/series_long.parquet", engine="pyarrow", partition_cols=partition_cols, index=False, compression="snappy")

# show summary
print(f"Saved {len(series_long)} rows to prepared/series_long.parquet partitioned by {partition_cols}")

KeyboardInterrupt: 